# Ungraded Lab: End-to-End ML Workflow with SageMaker

## Task 1: Training Data Preparation

<b>Step 1:</b>

Prepare and upload training data to S3 for SageMaker training job.

We provide the initial data loading:


In [ ]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import pandas as pd
import numpy as np
from sagemaker.sklearn.estimator import SKLearn
from sklearn.model_selection import train_test_split
from io import StringIO


# Load initial data
df = pd.read_csv('ticketwise_dataset.csv')

# Your code here:

# 1. Split data into training and validation sets

# 2. Upload two different files to S3 bucket: training_ticketwise_dataset and validation_ticketwise_dataset

We provide this helper function:

In [ ]:
def upload_df_to_s3(df, bucket, key):
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)
    s3_client.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())

## Task 2: Training Job Script and Configuration

<b>Step 1:</b>

Create the entry point script, named `train_ticketwise_regressor_model.py`, that SageMaker will use during the training process.

- Create a new file, named `train_ticketwise_regressor_model.py`, by selecting `New -> Python File`.
- This file should include all the code required to train your model.
- Include any necessary imports, data loading, model definition, training loop, and saving the trained model.
- Additionally, define the `model_fn` and `predict_fn` functions so SageMaker knows how to load the trained model and make predictions.

In [ ]:
# Add necessary imports

# ==========================================================
# TRAINING LOGIC (Instruction Version)
# ==========================================================

# Your code here:
# 1. Load the dataset from S3 into a pandas DataFrame.
#    Hint: Use boto3.client('s3') to fetch the CSV file and pd.read_csv to load it.
# 2. Select features and label for model training.
#    Example features: ["attachments", "avg_resolution_customer", "contract_value"]
#    Example label: "resolution_time"
# 3. Split your data into training and test sets.
#    Hint: Use sklearn.model_selection.train_test_split with test_size=0.2
# 4. Initialize a RandomForestRegressor and train it on your training set.
#    Hint: Set random_state=42 for reproducibility.
# 5. Save your trained model using joblib into '/opt/ml/model/model.joblib'.
#    Hint: SageMaker expects the model to be in this folder for deployment.

# ==========================================================
# HOSTING LOGIC
# ==========================================================

# Your code here:
# 1. Implement model_fn to load the model from the model directory when deploying.
#    Hint: Use joblib.load() on '/opt/ml/model/model.joblib'.
# 2. Implement predict_fn to make predictions with the loaded model.
#    Hint: Use model.predict(input_data) where input_data comes from the request.

<b>Step 2</b>:

Back to your notebook, create the SageMaker sklearn estimator :

In [ ]:
# Initialize SageMaker session
session = sagemaker.Session()
bucket = session.default_bucket()
prefix = "ticketwise-regressor-model"
role = get_execution_role()

sklearn_estimator = SKLearn(
    entry_point='train_ticketwise_regressor_model.py', # Add the training job name here
    role=role,
    instance_count=1,
    instance_type='ml.t3.large',
    framework_version="1.2-1",
    sagemaker_session=session,
    output_path=f"s3://{bucket}/{prefix}/output",

)

## Task 3: Training Job Execution

<b>Step 1:</b>

Launch and monitor the training job in SageMaker:

In [ ]:
# Initialize training job
print(f"output paht is s3://{bucket}/{prefix}/output")
# Kick off training job
sklearn_estimator.fit()

## Task 4: Model Deployment

<b>Step 1:</b>

Deploy trained model to SageMaker endpoint. We provide endpoint configuration template:

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

endpoint_name = #endpoint name

print(f"\nStarting model deployment to endpoint: {endpoint_name}...")
# The .deploy() method creates a real-time endpoint with the trained model.
# It provisions the infrastructure and makes the model available for requests.
predictor = sklearn_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
    endpoint_name=endpoint_name
)

print(f"Endpoint successfully deployed! Endpoint name: {predictor.endpoint_name}")

## Task 5: Endpoint Testing

<b> Step 1:</b>

Test deployed endpoint with sample tickets. We provide sample test data:

In [ ]:
# Your code here:
# Load CSV directly from S3
df_test_endpoint = #Your code here
print(df_test_endpoint.shape)

# Select features and label
features = #Select the same features used for model training
print(f"\nSending sample data to the endpoint: \n {df_test_endpoint[features]}")

# The .predict() method sends the data to the endpoint and returns the prediction.
prediction = predictor.predict(df[features].head())

print(f"\nPrediction received from the endpoint: {prediction}")

After creating and testing your endpoint, be sure to delete it to avoid incurring costs.

In [ ]:
print("Deleting the endpoint to avoid charges")

try:
    predictor.delete_endpoint()
    print("Endpoint successfully deleted.")
except Exception as e:
    print(f"Error deleting the endpoint. Please delete it manually in the SageMaker console. Error: {e}")

## Solution Code
Need a hand or are curious to compare your approach? Below is a complete solution you can use as a reference. This is just one of many valid ways to solve the problem. Make sure to give it a try on your own first. Use this implementation to troubleshoot, learn new techniques, or confirm your logic. Keep experimenting and enjoy the process!

## Task 1: Training Data Preparation Solution Code

<b>Step 1:</b>

Prepare and upload training data to S3 for SageMaker training job.

We provide the initial data loading:


In [ ]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import pandas as pd
import numpy as np
from sagemaker.sklearn.estimator import SKLearn
from sklearn.model_selection import train_test_split
from io import StringIO

# Load initial data
df = pd.read_csv('ticketwise_dataset.csv')

# 1. Split data into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# 2. Upload two different files to S3 bucket
BUCKET_NAME = 'ticketwise-pipeline'

# Initialize S3 client
s3_client = boto3.client('s3')

# Helper function to upload DataFrame to S3 as CSV
def upload_df_to_s3(df, bucket, key):
    csv_buffer = StringIO()
    df.to_csv(csv_buffer, index=False)
    s3_client.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())

# Upload training and validation datasets
upload_df_to_s3(train_df, BUCKET_NAME, 'training_ticketwise_dataset.csv')
upload_df_to_s3(val_df, BUCKET_NAME, 'validation_ticketwise_dataset.csv')

print("Training and validation datasets uploaded to S3 successfully!")

## Task 2: Training Job Script and Configuration Solution Code

<b>Step 1:</b>

Create the entry point script, named `train_ticketwise_regressor_model.py`, that SageMaker will use during the training process.

- Create a new python file, named `train_ticketwise_regressor_model.py`, by selecting `New -> Python File`.
- This file should include all the code required to train your model.
- Include any necessary imports, data loading, model definition, training loop, and saving the trained model.
- Additionally, define the `model_fn` and `predict_fn` functions so SageMaker knows how to load the trained model and make predictions.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import joblib
import os
import boto3
from io import StringIO
from sklearn.model_selection import train_test_split

# ==========================================================
# TRAINING LOGIC
# This part runs when you call estimator.fit()
# ==========================================================
if __name__ == '__main__':

    # 1. Load the dataset from S3 into a pandas DataFrame.
    # Read the training data
    # Download the Ticketwise data from S3
    BUCKET_NAME = 'ticketwise-pipeline'
    # Initialize S3 client
    s3_client = boto3.client('s3')
    response = s3_client.get_object(Bucket=BUCKET_NAME, Key="training_ticketwise_dataset.csv")
    # Read content into pandas DataFrame
    content = response['Body'].read().decode('utf-8')
    df = pd.read_csv(StringIO(content))

    # 2. Select features and label for model training.
    features = ["attachments", "avg_resolution_customer", "contract_value"]
    label = "resolution_time"

    X_train = df[features]
    y_train = df[label]

    # 4. Initialize a RandomForestRegressor and train it on your training set.
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    #  5. Save your trained model using joblib into '/opt/ml/model/model.joblib'.
    joblib.dump(model, os.path.join('/opt/ml/model', 'model.joblib'))


# ==========================================================
# HOSTING LOGIC (Functions needed for deployment)
# This part is used when you call estimator.deploy()
# ==========================================================

# 1. Implement model_fn to load the model from the model directory when deploying.
def model_fn(model_dir):
    """
    When you deploy, SageMaker calls this to load your model.
    model_dir is '/opt/ml/model'.
    """
    model = joblib.load(os.path.join(model_dir, "model.joblib"))
    return model

# 2. Implement predict_fn to make predictions with the loaded model.
def predict_fn(input_data, model):
    """
    SageMaker calls this to make a prediction.
    'input_data' is the data from your request, and 'model' is the
    model loaded by model_fn.
    """
    return model.predict(input_data)

<b> Step 2:</b>

Back to your notebook, create the SageMaker sklearn estimator:

In [ ]:
# Initialize SageMaker session
session = sagemaker.Session()
bucket = session.default_bucket()
prefix = "ticketwise-regressor-model"
role = get_execution_role()

sklearn_estimator = SKLearn(
    entry_point='train_ticketwise_regressor_model.py',
    role=role,
    instance_count=1,
    instance_type='ml.t3.xlarge',
    framework_version="1.2-1",
    sagemaker_session=session,
    output_path=f"s3://{bucket}/{prefix}/output")

## Task 3: Training Job Execution Solution Code

<b>Step 1:</b>

Launch and monitor the training job in SageMaker:

In [ ]:
# Initialize training job
print(f"output paht is s3://{bucket}/{prefix}/output")
# Kick off training job
sklearn_estimator.fit()

## Task 4: Model Deployment Solution Code

<b>Step 1:</b>

Deploy trained model to SageMaker endpoint. We provide endpoint configuration template:


In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

endpoint_name = "ticketwise-regressor-model-endpoint"

print(f"\nStarting model deployment to endpoint: {endpoint_name}...")
# The .deploy() method creates a real-time endpoint with the trained model.
# It provisions the infrastructure and makes the model available for requests.
predictor = sklearn_estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
    endpoint_name=endpoint_name
)

print(f"Endpoint successfully deployed! Endpoint name: {predictor.endpoint_name}")

## Task 5: Endpoint Testing Solution Code

<b>Step 1:</b>

Test deployed endpoint with sample tickets. We provide sample test data:

In [ ]:
# Load CSV directly from S3
df_test_endpoint = pd.read_csv(f"s3://{BUCKET_NAME}/validation_ticketwise_dataset.csv")
print(df_test_endpoint.shape)

# Select features and label
features = ["attachments", "avg_resolution_customer", "contract_value"]
print(f"\nSending sample data to the endpoint: \n {df_test_endpoint[features]}")

# The .predict() method sends the data to the endpoint and returns the prediction.
prediction = predictor.predict(df[features].head())

print(f"\nPrediction received from the endpoint: {prediction}")

After creating and testing your endpoint, be sure to delete it to avoid incurring costs.

In [ ]:
print("Deleting the endpoint to avoid charges")

try:
    predictor.delete_endpoint()
    print("Endpoint successfully deleted.")
except Exception as e:
    print(f"Error deleting the endpoint. Please delete it manually in the SageMaker console. Error: {e}")